# 🧹 Notebook 01 — Data Cleaning Pipeline
### Senti-Recommend: A Hybrid AI Tool Discovery Engine
**PGD Data Science with AI — Postgraduate Project**

---

## 📋 Purpose of This Notebook
This notebook loads the two raw data files that form the backbone of the Senti-Recommend project:
- `ai_tools_metadata.csv` — 482 AI tools with ratings, installs, pricing and descriptions
- `ai_tools_reviews.csv` — 69,727 user reviews across those tools

We perform structured cleaning on both files and engineer new features that will be used in later notebooks for:
- Sentiment Analysis (Notebook 02)
- Collaborative Filtering (Notebook 03)
- Hybrid Scoring and the Streamlit App (Notebook 04)

---

## 📁 Outputs
| File | Description |
|------|-------------|
| `cleaned_metadata.csv` | Cleaned tool metadata with engineered features |
| `cleaned_reviews.csv` | Cleaned review text with quality flags |

---

## 🗂️ Notebook Structure
1. Import Libraries
2. Load Raw Data
3. Initial Diagnostic
4. Clean Metadata
5. Clean Reviews
6. Post-Cleaning Validation
7. Save Cleaned Files
8. Summary

---
## 1. Import Libraries
We use three standard Python libraries:
- **pandas** — for loading and manipulating tabular data
- **numpy** — for numerical operations (e.g. handling divide-by-zero safely)
- **warnings** — to suppress non-critical output so the notebook stays readable

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

print('✅ Libraries imported successfully')

✅ Libraries imported successfully


---
## 2. Load Raw Data
We load both CSV files using `encoding='utf-8-sig'` to handle the BOM (Byte Order Mark) character
that was present at the start of both files. Without this, the first column name would appear
as `\ufeffapp_id` instead of `app_id`.

In [2]:
# ── Load both raw files ───────────────────────────────────────────────────────
meta    = pd.read_csv('../data/raw/ai_tools_metadata.csv', encoding='utf-8-sig')
reviews = pd.read_csv('../data/raw/ai_tools_reviews.csv',  encoding='utf-8-sig')

print(f'Metadata  shape : {meta.shape}    → {meta.shape[0]} tools,   {meta.shape[1]} columns')
print(f'Reviews   shape : {reviews.shape} → {reviews.shape[0]:,} reviews, {reviews.shape[1]} columns')

Metadata  shape : (482, 30)    → 482 tools,   30 columns
Reviews   shape : (69727, 13) → 69,727 reviews, 13 columns


---
## 3. Initial Diagnostic
Before touching the data, we run a full diagnostic to understand:
- **Missing values** — which columns have gaps and how many
- **Data types** — are numbers stored as text? are dates readable?
- **Duplicates** — are any rows repeated?
- **Value distributions** — what categories and ratings exist?

Think of this like a doctor's check-up before prescribing treatment.

In [3]:
# ── Metadata diagnostic ───────────────────────────────────────────────────────
print('=' * 55)
print('METADATA — MISSING VALUES')
print('=' * 55)
missing_meta = meta.isnull().sum()
print(missing_meta[missing_meta > 0].to_string())

print('\n' + '=' * 55)
print('METADATA — DATA TYPES')
print('=' * 55)
print(meta.dtypes.to_string())

print('\n' + '=' * 55)
print(f'METADATA — DUPLICATE ROWS: {meta.duplicated().sum()}')
print('=' * 55)

print('\nPricing model breakdown:')
print(meta['pricing_model'].value_counts().to_string())

print('\nProject category breakdown:')
print(meta['project_category'].value_counts().to_string())

METADATA — MISSING VALUES
android_version    482
updated_on          34
released_on          3
size               482

METADATA — DATA TYPES
app_id               object
app_name             object
developer            object
category_scraped     object
project_category     object
avg_rating          float64
total_ratings         int64
total_reviews         int64
installs              int64
description          object
summary              object
content_rating       object
android_version     float64
app_version          object
updated_on          float64
released_on          object
size                float64
pricing_model        object
price_usd           float64
offers_iap             bool
free                   bool
url                  object
icon_url             object
genre_id             object
ratings_1_star        int64
ratings_2_star        int64
ratings_3_star        int64
ratings_4_star        int64
ratings_5_star        int64
scraped_at           object

METADATA — DUPLICA

In [4]:
# ── Reviews diagnostic ────────────────────────────────────────────────────────
print('=' * 55)
print('REVIEWS — MISSING VALUES')
print('=' * 55)
missing_rev = reviews.isnull().sum()
print(missing_rev[missing_rev > 0].to_string())

print('\n' + '=' * 55)
print('REVIEWS — DATA TYPES')
print('=' * 55)
print(reviews.dtypes.to_string())

print('\n' + '=' * 55)
print(f'REVIEWS — DUPLICATE ROWS: {reviews.duplicated().sum()}')
print('=' * 55)

print('\nStar rating distribution:')
print(reviews['star_rating'].value_counts().sort_index().to_string())

REVIEWS — MISSING VALUES
user_name                 2
review_text               1
review_title          69727
reply_text            48355
reply_date            48355
app_version_review     9867

REVIEWS — DATA TYPES
app_id                 object
review_id              object
user_name              object
review_text            object
review_title          float64
star_rating             int64
thumbs_up_count         int64
review_date            object
reply_text             object
reply_date             object
app_version_review     object
sort_source            object
scraped_at             object

REVIEWS — DUPLICATE ROWS: 0

Star rating distribution:
star_rating
1    18058
2     4567
3     4927
4     7363
5    34812


---
## 4. Clean Metadata
We apply cleaning in a logical sequence — first remove what we don't need, then fix what
we keep, then engineer new features on top of the clean base.

### Steps in this section:
| Step | Action | Reason |
|------|--------|--------|
| 4.1 | Drop irrelevant columns | 6 columns are either 100% missing or not useful for our model |
| 4.2 | Fix date columns | Dates stored as Unix timestamps or inconsistent strings |
| 4.3 | Fill missing dates | 34 rows missing `updated_on`, 3 missing `released_on` |
| 4.4 | Standardise text | Strip whitespace from category and pricing columns |
| 4.5 | Clip rating values | Ensure `avg_rating` is between 1.0 and 5.0 |
| 4.6 | Engineer `bayesian_avg` | A fairer rating score that accounts for review volume |
| 4.7 | Engineer `review_rate` | Reviews per install — a proxy for user engagement strength |
| 4.8 | Fill missing `price_usd` | Free tools have NaN here — should be 0.0 |

In [5]:
# ── 4.1  Drop irrelevant / fully-empty columns ────────────────────────────────
# android_version and size are 100% missing.
# icon_url, url, genre_id, content_rating are not needed for the recommender.

cols_to_drop = ['android_version', 'size', 'icon_url', 'content_rating',
                'genre_id', 'url']
meta.drop(columns=cols_to_drop, inplace=True)

print(f'Columns after drop: {meta.shape[1]}  (removed {len(cols_to_drop)} columns)')
print(list(meta.columns))

Columns after drop: 24  (removed 6 columns)
['app_id', 'app_name', 'developer', 'category_scraped', 'project_category', 'avg_rating', 'total_ratings', 'total_reviews', 'installs', 'description', 'summary', 'app_version', 'updated_on', 'released_on', 'pricing_model', 'price_usd', 'offers_iap', 'free', 'ratings_1_star', 'ratings_2_star', 'ratings_3_star', 'ratings_4_star', 'ratings_5_star', 'scraped_at']


In [6]:
# ── 4.2  Fix date columns ─────────────────────────────────────────────────────
# updated_on was stored as a Unix epoch float (seconds since 1970-01-01).
# released_on and scraped_at are standard date strings — pandas can parse them.

meta['updated_on']  = pd.to_datetime(meta['updated_on'], unit='s', errors='coerce')
meta['released_on'] = pd.to_datetime(meta['released_on'], errors='coerce')
meta['scraped_at']  = pd.to_datetime(meta['scraped_at'],  errors='coerce')

print('Sample updated_on values after conversion:')
print(meta['updated_on'].dropna().head(3).to_string())

Sample updated_on values after conversion:
0   2026-02-09 15:23:04
1   2026-02-10 18:26:24
2   2026-02-17 11:05:14


In [7]:
# ── 4.3  Fill missing dates using median ─────────────────────────────────────
# We use the median (middle value) rather than mean because dates are ordinal.
# The median is more robust to outliers (e.g. very old or recently updated apps).

median_updated  = meta['updated_on'].dropna().sort_values().iloc[
                      len(meta['updated_on'].dropna()) // 2]
median_released = meta['released_on'].dropna().sort_values().iloc[
                      len(meta['released_on'].dropna()) // 2]

meta['updated_on']  = meta['updated_on'].fillna(median_updated)
meta['released_on'] = meta['released_on'].fillna(median_released)

print(f'Median updated_on  used for fill : {median_updated.date()}')
print(f'Median released_on used for fill : {median_released.date()}')
print(f'Missing updated_on  remaining    : {meta["updated_on"].isnull().sum()}')
print(f'Missing released_on remaining    : {meta["released_on"].isnull().sum()}')

Median updated_on  used for fill : 2026-01-30
Median released_on used for fill : 2023-08-08
Missing updated_on  remaining    : 0
Missing released_on remaining    : 0


In [8]:
# ── 4.4  Standardise text columns ────────────────────────────────────────────
# .str.strip() removes leading/trailing spaces.
# This prevents 'Freemium ' and 'Freemium' being treated as different categories.

for col in ['app_name', 'developer', 'project_category', 'pricing_model']:
    meta[col] = meta[col].str.strip()

print('Pricing model values after strip:')
print(meta['pricing_model'].value_counts().to_string())

Pricing model values after strip:
pricing_model
Freemium    419
Free         61
Paid          2


In [9]:
# ── 4.5  Clip avg_rating to valid range [1, 5] ────────────────────────────────
# In theory ratings should already be in [1,5], but we clip defensively.
# clip(1, 5) means: any value below 1 becomes 1, any above 5 becomes 5.

meta['avg_rating'] = meta['avg_rating'].clip(1, 5)

print('avg_rating stats after clipping:')
print(meta['avg_rating'].describe().round(4).to_string())

avg_rating stats after clipping:
count    482.0000
mean       4.2585
std        0.4905
min        1.8000
25%        4.0756
50%        4.3798
75%        4.5800
max        5.0000


In [10]:
# ── 4.6  Engineer: Bayesian Average Rating ────────────────────────────────────
#
# PROBLEM: A tool with 3 reviews all rated 5★ would rank above a tool with
# 50,000 reviews averaging 4.8★. That's misleading.
#
# SOLUTION: The Bayesian average 'shrinks' extreme ratings toward the global
# mean when a tool has very few reviews.
#
# Formula:
#   bayesian_avg = (v / (v + m)) * R  +  (m / (v + m)) * C
#
# Where:
#   v = number of ratings for this tool
#   m = median number of ratings across ALL tools (our 'confidence threshold')
#   R = this tool's average rating
#   C = global average rating across all tools

m = meta['total_ratings'].median()   # confidence threshold
C = meta['avg_rating'].mean()        # global mean rating

meta['bayesian_avg'] = (
    (meta['total_ratings'] / (meta['total_ratings'] + m)) * meta['avg_rating']
    + (m / (meta['total_ratings'] + m)) * C
).round(4)

print(f'Global mean rating (C) : {C:.4f}')
print(f'Median total_ratings (m): {m:.0f}')
print('\nComparison — avg_rating vs bayesian_avg (first 8 rows):')
print(meta[['app_name','total_ratings','avg_rating','bayesian_avg']].head(8).to_string(index=False))

Global mean rating (C) : 4.2585
Median total_ratings (m): 9458

Comparison — avg_rating vs bayesian_avg (first 8 rows):
                      app_name  total_ratings  avg_rating  bayesian_avg
ChatOn - AI Chat Bot Assistant         415843    4.530914        4.5249
             AI Chatbot - Nova        2099272    3.817843        3.8198
 Chatbot AI - Search Assistant         853354    4.386683        4.3853
 AI Chat・Ask Chatbot Assistant         184977    4.497138        4.4855
ChatBox: AI Chat Bot Assistant          95187    4.496429        4.4749
PolyBuzz: Chat with AI Friends         905661    3.800876        3.8056
  Chatbot - AI Smart Assistant         152569    4.455918        4.4444
 Ask AI - Chat with AI Chatbot        1097214    4.443531        4.4420


In [11]:
# ── 4.7  Engineer: Review Rate ────────────────────────────────────────────────
#
# review_rate = total_reviews / installs
#
# This tells us: 'Of everyone who installed this tool, what fraction bothered
# to write a review?'  High review rate = strong feelings (positive OR negative).
# This is useful for detecting 'hidden gems' — tools people feel strongly about
# despite low total install counts.
#
# We use .replace(0, np.nan) to avoid dividing by zero for tools with 0 installs.

meta['review_rate'] = (
    meta['total_reviews'] / meta['installs'].replace(0, np.nan)
).round(6)

print('review_rate stats:')
print(meta['review_rate'].describe().round(6).to_string())
print('\nTop 5 most reviewed per install:')
print(meta[['app_name','total_reviews','installs','review_rate']]
      .sort_values('review_rate', ascending=False)
      .head(5)
      .to_string(index=False))

review_rate stats:
count    482.000000
mean       0.000700
std        0.001400
min        0.000000
25%        0.000109
50%        0.000263
75%        0.000696
max        0.015317

Top 5 most reviewed per install:
                      app_name  total_reviews  installs  review_rate
   GoMind AI: AI Tasks & Notes              7       457     0.015317
Pose: AI Video Maker, AI Photo           9102    772297     0.011786
  SuperImage Pro - AI Enhancer            103     10872     0.009474
        Fieldy - AI note taker             79      9751     0.008102
MonetizeAI: Make Money with AI            110     13706     0.008026


In [12]:
# ── 4.8  Fill missing price_usd with 0.0 ─────────────────────────────────────
# Free tools have NaN in price_usd. A missing price for a free tool means £0,
# not 'unknown', so we fill with 0.0 explicitly.

meta['price_usd'] = meta['price_usd'].fillna(0.0)

print('price_usd missing after fill:', meta['price_usd'].isnull().sum())
print('price_usd unique values:', sorted(meta['price_usd'].unique()))

price_usd missing after fill: 0
price_usd unique values: [np.float64(0.0), np.float64(2.99), np.float64(3.49)]


---
## 5. Clean Reviews
The reviews file contains the raw text we will analyse for sentiment. Clean text = better NLP results.

### Steps in this section:
| Step | Action | Reason |
|------|--------|--------|
| 5.1 | Drop irrelevant columns | `review_title` is 100% empty; others not needed for NLP |
| 5.2 | Drop row with missing review text | 1 row — can't analyse what isn't there |
| 5.3 | Fill missing user names | 2 rows — replace NaN with 'Anonymous' |
| 5.4 | Fix date columns | Convert string dates to proper datetime format |
| 5.5 | Clip star ratings | Ensure values are between 1 and 5 |
| 5.6 | Clean review text | Remove non-printable characters, collapse whitespace |
| 5.7 | Engineer `review_length` | Useful quality signal for the NLP step |
| 5.8 | Engineer `is_short_review` | Flag reviews under 20 chars as too short for sentiment |
| 5.9 | Engineer `has_reply` | Whether the developer responded — trust signal |

In [13]:
# ── 5.1  Drop irrelevant columns ──────────────────────────────────────────────
# review_title : 100% empty (69,727 nulls from 69,727 rows)
# app_version_review : version strings — not useful for NLP or recommendations
# sort_source : internal scraping metadata — not relevant to analysis

reviews.drop(columns=['review_title', 'app_version_review', 'sort_source'],
             inplace=True)

print('Remaining review columns:', list(reviews.columns))

Remaining review columns: ['app_id', 'review_id', 'user_name', 'review_text', 'star_rating', 'thumbs_up_count', 'review_date', 'reply_text', 'reply_date', 'scraped_at']


In [14]:
# ── 5.2  Drop the single row missing review_text ──────────────────────────────
before = len(reviews)
reviews = reviews.dropna(subset=['review_text']).copy()
after  = len(reviews)

print(f'Rows removed: {before - after}  (expected: 1)')
print(f'Reviews remaining: {after:,}')

Rows removed: 1  (expected: 1)
Reviews remaining: 69,726


In [15]:
# ── 5.3  Fill missing user_name with 'Anonymous' ──────────────────────────────
reviews['user_name'] = reviews['user_name'].fillna('Anonymous')

print('Missing user_name after fill:', reviews['user_name'].isnull().sum())

Missing user_name after fill: 0


In [16]:
# ── 5.4  Fix date columns ─────────────────────────────────────────────────────
for col in ['review_date', 'reply_date', 'scraped_at']:
    reviews[col] = pd.to_datetime(reviews[col], errors='coerce')

print('Sample review_date values:')
print(reviews['review_date'].dropna().head(3).to_string())

Sample review_date values:
0   2026-02-03 20:36:24
1   2026-01-08 03:40:33
2   2024-09-09 23:16:54


In [17]:
# ── 5.5  Clip star_rating to valid range [1, 5] ───────────────────────────────
reviews['star_rating'] = reviews['star_rating'].clip(1, 5)

print('Star rating distribution after clip:')
print(reviews['star_rating'].value_counts().sort_index().to_string())

Star rating distribution after clip:
star_rating
1    18057
2     4567
3     4927
4     7363
5    34812


In [18]:
# ── 5.6  Clean review_text ────────────────────────────────────────────────────
# Step A: .str.strip()          — remove leading/trailing whitespace
# Step B: replace non-ASCII     — emojis and special chars become spaces
#                                 (VADER and TextBlob work best on plain text)
# Step C: collapse whitespace   — multiple spaces become one

reviews['review_text'] = (
    reviews['review_text']
    .str.strip()
    .str.replace(r'[^\x20-\x7E\n]', ' ', regex=True)  # keep printable ASCII only
    .str.replace(r'\s+', ' ',        regex=True)        # collapse whitespace
)

print('Sample cleaned review:')
print(reviews['review_text'].iloc[2][:300])

Sample cleaned review:
it's a little tricky sometimes to get the right wording of prompts, but once you figure it out it gets easier and easier to generate what you want. For images, it seems to not be as good as dedicated image generators. Overall I have been enjoying my experience. It's cool to figure out unexpected way


In [19]:
# ── 5.7  Engineer: review_length ─────────────────────────────────────────────
# Simply counts the number of characters in each review.
# Very short reviews ('Great!', 'Bad') carry little sentiment information.
# We'll use this to filter or weight reviews in the NLP notebook.

reviews['review_length'] = reviews['review_text'].str.len()

print('Review length statistics:')
print(reviews['review_length'].describe().round(1).to_string())

Review length statistics:
count    69726.0
mean       162.0
std        171.0
min          1.0
25%         17.0
50%         85.0
75%        295.0
max       2954.0


In [20]:
# ── 5.8  Engineer: is_short_review flag ──────────────────────────────────────
# Reviews under 20 characters (e.g. 'Great app!', 'Trash') are flagged.
# We keep them — but flag them so Notebook 02 can optionally filter them.

reviews['is_short_review'] = reviews['review_length'] < 20

short_count = reviews['is_short_review'].sum()
pct = short_count / len(reviews) * 100
print(f'Short reviews flagged : {short_count:,}  ({pct:.1f}% of total)')
print('\nExamples of short reviews:')
print(reviews[reviews['is_short_review']]['review_text'].head(5).to_string())

Short reviews flagged : 18,941  (27.2% of total)

Examples of short reviews:
101    It fantastic
102           sucks
103       very good
106            good
107     ok aahe app


In [21]:
# ── 5.9  Engineer: has_reply flag ────────────────────────────────────────────
# True  = the developer wrote a reply to this review
# False = no reply
# Developer responsiveness can be used as a qualitative trust signal in the app.

reviews['has_reply'] = reviews['reply_text'].notna()

reply_count = reviews['has_reply'].sum()
print(f'Reviews with a developer reply: {reply_count:,}  '
      f'({reply_count/len(reviews)*100:.1f}%)')

Reviews with a developer reply: 21,371  (30.6%)


---
## 6. Post-Cleaning Validation
We recheck both DataFrames to confirm:
- No unexpected missing values remain in critical columns
- Shapes are as expected
- New engineered columns are present and correctly typed

In [22]:
# ── Validate Metadata ─────────────────────────────────────────────────────────
print('METADATA — Final Shape:', meta.shape)
print('\nAll columns:')
print(list(meta.columns))

print('\nMissing values (should be empty):')
remaining = meta.isnull().sum()
print(remaining[remaining > 0].to_string() or '  ✅ None!')

print('\nNew engineered columns preview:')
print(meta[['app_name','avg_rating','bayesian_avg','review_rate']]
      .head(6).to_string(index=False))

METADATA — Final Shape: (482, 26)

All columns:
['app_id', 'app_name', 'developer', 'category_scraped', 'project_category', 'avg_rating', 'total_ratings', 'total_reviews', 'installs', 'description', 'summary', 'app_version', 'updated_on', 'released_on', 'pricing_model', 'price_usd', 'offers_iap', 'free', 'ratings_1_star', 'ratings_2_star', 'ratings_3_star', 'ratings_4_star', 'ratings_5_star', 'scraped_at', 'bayesian_avg', 'review_rate']

Missing values (should be empty):
Series([], )

New engineered columns preview:
                      app_name  avg_rating  bayesian_avg  review_rate
ChatOn - AI Chat Bot Assistant    4.530914        4.5249     0.000166
             AI Chatbot - Nova    3.817843        3.8198     0.000278
 Chatbot AI - Search Assistant    4.386683        4.3853     0.000082
 AI Chat・Ask Chatbot Assistant    4.497138        4.4855     0.000127
ChatBox: AI Chat Bot Assistant    4.496429        4.4749     0.000237
PolyBuzz: Chat with AI Friends    3.800876        3.8056  

In [23]:
# ── Validate Reviews ──────────────────────────────────────────────────────────
print('REVIEWS — Final Shape:', reviews.shape)
print('\nAll columns:')
print(list(reviews.columns))

print('\nMissing values in key columns:')
key_cols = ['app_id','user_name','review_text','star_rating',
            'review_date','review_length','is_short_review','has_reply']
print(reviews[key_cols].isnull().sum().to_string())

print('\nNew engineered columns preview:')
print(reviews[['user_name','star_rating','review_length',
               'is_short_review','has_reply']].head(6).to_string(index=False))

REVIEWS — Final Shape: (69726, 13)

All columns:
['app_id', 'review_id', 'user_name', 'review_text', 'star_rating', 'thumbs_up_count', 'review_date', 'reply_text', 'reply_date', 'scraped_at', 'review_length', 'is_short_review', 'has_reply']

Missing values in key columns:
app_id             0
user_name          0
review_text        0
star_rating        0
review_date        0
review_length      0
is_short_review    0
has_reply          0

New engineered columns preview:
          user_name  star_rating  review_length  is_short_review  has_reply
      Leafand Lotus            1            373            False       True
        Hayden Winn            1            490            False      False
               Jenn            5            494            False      False
    Curtis Majeskie            1            421            False      False
Coldwater (Fauxtog)            1            394            False       True
   Kennedy Byington            5            496            False      

---
## 7. Save Cleaned Files
We save both cleaned DataFrames as CSV files. Using `index=False` prevents pandas
from writing a redundant row-number column into the file.

In [24]:
# ── Save cleaned files ────────────────────────────────────────────────────────
meta.to_csv('cleaned_metadata.csv',    index=False)
reviews.to_csv('cleaned_reviews.csv',  index=False)

print('✅ cleaned_metadata.csv saved')
print('✅ cleaned_reviews.csv  saved')

✅ cleaned_metadata.csv saved
✅ cleaned_reviews.csv  saved


---
## 8. Summary

### What was cleaned

#### Metadata (482 tools)
| Action | Detail |
|--------|--------|
| Dropped 6 columns | `android_version`, `size` (100% missing); `icon_url`, `url`, `genre_id`, `content_rating` (not needed) |
| Fixed date formats | `updated_on` converted from Unix timestamp; all 3 date columns now proper datetime |
| Filled 34 missing dates | `updated_on` — filled with column median date |
| Filled 3 missing dates | `released_on` — filled with column median date |
| Filled price NaN | `price_usd` → 0.0 for free tools |
| Clipped ratings | `avg_rating` constrained to [1, 5] |
| **New: `bayesian_avg`** | Fairer rating that penalises tools with very few reviews |
| **New: `review_rate`** | Engagement signal: reviews per install |

#### Reviews (69,726 reviews)
| Action | Detail |
|--------|--------|
| Dropped 3 columns | `review_title` (100% empty), `app_version_review`, `sort_source` |
| Removed 1 row | Missing `review_text` |
| Filled 2 missing names | `user_name` → 'Anonymous' |
| Cleaned text | Non-printable characters removed; whitespace collapsed |
| **New: `review_length`** | Character count — NLP quality signal |
| **New: `is_short_review`** | True if < 20 chars — 18,941 flagged |
| **New: `has_reply`** | True if developer replied — 21,371 flagged |

---

### ➡️ Next: Notebook 02 — Sentiment Analysis
We will apply **VADER** (Valence Aware Dictionary and sEntiment Reasoner) to each
cleaned review to produce a sentiment score between -1.0 (very negative) and +1.0
(very positive). These scores will later be combined with the collaborative filtering
ratings to produce the hybrid recommendation score.